# v8 GQA M-packing — the gate (Cut 1: CUDA-core, vast.ai T4)

v8 packs the `G = H_q/H_kv` query heads that share one KV head into the score GEMM's **M** dimension: a CTA reads that KV head **once** and runs **G query rows** against it (G warps active, not 1), so decode `AI = 2/b -> 2G/b`. This is Cut 1 — CUDA-core, sm_75/T4 — isolating the M-packing variable cheaply. **Gate 1 = correctness (v8 + v7 regression); Gate 2 = the quiz.**

## 0. Dependencies + GPU (venv-safe)

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes --
#    importing torch first on a numpy-less venv (vast.ai) prints 'Failed to initialize NumPy'.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell (the old CPU torch stays loaded until restart).')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Roofline first — decode `AI = 2G/b` (the per-step gate, recorded BEFORE the bench)

Predict on **sm_80 (A100, the Cut-2 perf target)**: AI rises `G×`, the HBM floor drops `G×`, but the limiter **stays HBM** (A100 fp16 ridge = 153, so even G=8's AI=8 is far below). The win is per-CTA efficiency, not a limiter flip — that's what the bench must show.

In [ ]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_80')   # AI is arch-independent; the t_hbm floor is the A100 target's
print(f"{'G':>3} | {'AI=2G/b':>8} | {'limiter':>7} | {'ridge':>6} | {'t_hbm floor':>12}")
for G in (1, 2, 4, 8, 16, 32):
    e = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=G)
    print(f'{G:>3} | {e.arithmetic_intensity:8.1f} | {e.limiter.upper():>7} | {e.ridge:6.1f} | {e.t_hbm*1e3:9.4f}ms')
print()
print('Prediction: AI rises Gx, floor drops Gx, limiter STAYS HBM (AI < ridge 153 even at G=8).')
print('v8 win = per-CTA efficiency (G warps + KV read once), NOT a limiter flip. Measured below.')


## 3. Build v8 (JIT)

In [ ]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v8_gqa')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
mod = build_kernel('v8_gqa')
print('built:', mod)


## 4. Correctness gate — v8 GQA + v7 regression  *(Gate 1 of 2)*

v8's GQA cases (decode `G in {1,2,4,8}`, non-multiple `N_k`, causal+offset, idle-warp `G=3` + multi-tile `G=16`, square reduction) **and** v7 unchanged. Oracle = `sdpa_reference_gqa` (KV expanded by `repeat_interleave(G)`).

In [ ]:
!python -m pytest tests/test_correctness.py -k "v8_gqa or v7_paged" -q


## 5. Decode benchmark — GQA `G=8` canonical (non-causal + causal)

`--heads 8 --gqa-group 8` -> H_q=8 query heads, H_kv=1 KV head. `vs naive` here is **v7 on the same attention with KV expanded to H_q heads** (the no-M-packing floor) — the clean same-session isolation of M-packing's one variable. `vs sdpa` is the torch GQA baseline.

In [ ]:
!python -m bench.harness --backend v8_gqa --decode --heads 8 --gqa-group 8
print()
!python -m bench.harness --backend v8_gqa --decode --heads 8 --gqa-group 8 --causal


## 6. The G-sweep — GEMV->GEMM per-CTA efficiency (the v8 deliverable)

`--heads 32` fixes H_q=32; `H_kv = 32//G` shrinks as G grows (KV bytes drop by G). Watch `us/tok` fall and `vs naive` (= vs v7 no-packing) rise as G activates G warps + reads KV once. The prediction-vs-measured curve as G crosses the M<16->M>=16 threshold (the tensor-core line is Cut 2 / A100).

In [ ]:
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32


## 7. Reclaim SDPA at batch — the headline the v7 data created

v7 **lost** to torch SDPA at B>=8 (0.5x). With G=8 (8 warps active, KV read once) v8 must reclaim the serving-batch regime. Hold G=8, sweep B; `vs sdpa` is the column that matters.

In [ ]:
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
